# 🚚 Анализ логистических маршрутов
## Интерактивная демонстрация поиска узких мест

**Цель:** Найти склады с аномально долгим временем обработки посылок и визуализировать результаты.

###  1. Импорт библиотек и подключение к БД

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

conn = sqlite3.connect('../logistics.db')
print("✅ Подключение к БД установлено")

###  2. Общая статистика по данным

In [ ]:
query_stats = """
SELECT 
    COUNT(DISTINCT parcel_id) as total_parcels,
    MIN(event_timestamp) as first_event,
    MAX(event_timestamp) as last_event,
    COUNT(*) as total_events,
    ROUND(AVG(weight_kg), 2) as avg_weight_kg
FROM logistics_log
"""
df_stats = pd.read_sql_query(query_stats, conn)
df_stats

###  3. Пример сырых данных

In [ ]:
df_sample = pd.read_sql_query("SELECT * FROM logistics_log LIMIT 10", conn)
df_sample

###  4. Распределение типов событий

In [ ]:
query_events = """
SELECT 
    event_type,
    COUNT(*) as count
FROM logistics_log
GROUP BY event_type
ORDER BY count DESC
"""
df_events = pd.read_sql_query(query_events, conn)

fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette("viridis", len(df_events))
wedges, texts, autotexts = ax.pie(
    df_events['count'], 
    labels=df_events['event_type'], 
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    textprops={'fontsize': 12}
)
ax.set_title('Распределение типов событий в логистике', fontsize=14, fontweight='bold', pad=20)
plt.show()

###  5. ТОП проблемных складов по времени обработки

In [ ]:
query_bottlenecks = """
SELECT 
    location_name as warehouse,
    parcels_processed,
    avg_processing_hours,
    max_processing_hours,
    delayed_over_24h
FROM v_bottlenecks
ORDER BY avg_processing_hours DESC
"""
df_bottlenecks = pd.read_sql_query(query_bottlenecks, conn)
df_bottlenecks

In [ ]:
# Визуализация топ-8 проблемных складов
fig, ax = plt.subplots(figsize=(14, 7))

top8 = df_bottlenecks.head(8).sort_values('avg_processing_hours', ascending=True)
colors = plt.cm.RdYlGn_r(top8['avg_processing_hours'] / top8['avg_processing_hours'].max())
bars = ax.barh(top8['warehouse'], top8['avg_processing_hours'], color=colors)

ax.axvline(x=8, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Норматив (8 часов)')
ax.bar_label(bars, fmt='%.1f ч', padding=5, fontsize=11, fontweight='bold')

ax.set_xlabel('Среднее время обработки (часы)', fontsize=13)
ax.set_ylabel('')
ax.set_title('ТОП-8 ПРОБЛЕМНЫХ ЛОГИСТИЧЕСКИХ ЦЕНТРОВ\nПо среднему времени простоя посылок', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='lower right')
ax.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

###  6. ТОП-10 самых долгих доставок (от создания до вручения)

In [ ]:
query_longest = """
WITH delivery_times AS (
    SELECT 
        parcel_id,
        MIN(CASE WHEN event_type = 'CREATED' THEN event_timestamp END) as created,
        MAX(CASE WHEN event_type = 'DELIVERED' THEN event_timestamp END) as delivered
    FROM logistics_log
    GROUP BY parcel_id
    HAVING delivered IS NOT NULL
)
SELECT 
    parcel_id,
    ROUND((JULIANDAY(delivered) - JULIANDAY(created)) * 24, 1) as total_hours,
    created,
    delivered
FROM delivery_times
ORDER BY total_hours DESC
LIMIT 10
"""
df_longest = pd.read_sql_query(query_longest, conn)
df_longest

###  7. Активность по складам (прибытия и отправления)

In [ ]:
query_location_activity = """
SELECT 
    location_name,
    event_type,
    COUNT(*) as count
FROM logistics_log
WHERE event_type IN ('ARRIVED', 'DEPARTED')
GROUP BY location_name, event_type
ORDER BY location_name
"""
df_activity = pd.read_sql_query(query_location_activity, conn)

pivot = df_activity.pivot(index='location_name', columns='event_type', values='count')
pivot = pivot.sort_values('ARRIVED', ascending=False)

ax = pivot.plot(kind='bar', figsize=(14, 6), width=0.8)
ax.set_title('Количество прибытий и отправлений по логистическим центрам', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('')
ax.set_ylabel('Количество событий', fontsize=12)
ax.legend(title='Тип события')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', linestyle='--', alpha=0.5)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', fontsize=9)

plt.tight_layout()
plt.show()

###  8. Тепловая карта загрузки по дням недели

In [ ]:
query_heatmap = """
SELECT 
    location_name,
    CASE strftime('%w', event_timestamp)
        WHEN '0' THEN 'Вс'
        WHEN '1' THEN 'Пн'
        WHEN '2' THEN 'Вт'
        WHEN '3' THEN 'Ср'
        WHEN '4' THEN 'Чт'
        WHEN '5' THEN 'Пт'
        WHEN '6' THEN 'Сб'
    END as day_of_week,
    COUNT(*) as volume
FROM logistics_log
WHERE event_type = 'ARRIVED'
GROUP BY location_name, day_of_week
"""
df_heatmap = pd.read_sql_query(query_heatmap, conn)

pivot = df_heatmap.pivot(index='location_name', columns='day_of_week', values='volume')
days_order = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
pivot = pivot.reindex(columns=days_order)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    pivot, 
    annot=True, 
    fmt='.0f', 
    cmap='YlOrRd', 
    ax=ax, 
    linewidths=0.5, 
    cbar_kws={'label': 'Количество прибытий'},
    annot_kws={'fontsize': 10}
)
ax.set_title('ТЕПЛОВАЯ КАРТА ЗАГРУЗКИ СКЛАДОВ ПО ДНЯМ НЕДЕЛИ', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('День недели', fontsize=12)
ax.set_ylabel('')

plt.tight_layout()
plt.show()

###  9. Выводы и рекомендации

На основе проведённого анализа можно сделать следующие **выводы**:

---

#### 🔴 Проблемные склады (узкие места)
- **Нижний Новгород-Волга** — среднее время обработки превышает **15 часов** (при нормативе 8 часов).
- **Самара-ЛЦ** — также показывает аномально высокие задержки.

#### 🟡 Пиковые нагрузки
- Наибольший объём прибытий приходится на **понедельник и вторник**.
- Рекомендуется усилить смены в начале недели.

#### 🟢 Рекомендации
1. Провести аудит сортировочного оборудования в Нижнем Новгороде и Самаре.
2. Пересмотреть штатное расписание на пиковые дни (Пн-Вт).
3. Внедрить мониторинг времени обработки в реальном времени.
4. Рассмотреть возможность перенаправления части потока на менее загруженные склады (Москва-Хаб, Санкт-Петербург).

---

**Инструменты:** Python (Pandas, Matplotlib, Seaborn), SQL (SQLite, оконные функции, CTE).

In [ ]:
# Закрываем соединение с БД
conn.close()
print("🔌 Соединение с БД закрыто")